In [ ]:
import pandas as pd
import os
from pathlib import Path
from google.colab import drive

# 1. Montar Google Drive
drive.mount('/content/drive')




Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:

!pip install biopython -q

In [ ]:
from Bio.PDB import PDBParser
archivo = "/content/drive/MyDrive/farmacos/Farmaco/TP FINAL/4L9M.pdb"
archivo2 = "/content/drive/MyDrive/farmacos/Farmaco/TP FINAL/1JUQ.pdb"
archivo3 = "/content/drive/MyDrive/farmacos/Farmaco/TP FINAL/5J3V.pdb"

# Ruta del PDB en Drive
pdb_file = archivo

# Residuos del pocket que dio P2Rank
pocket_residues = [
    219, 220, 223, 246, 249, 250,
    414, 417, 418, 420, 461, 462,
    463, 466, 467, 470, 473
]

parser = PDBParser(QUIET=True)
structure = parser.get_structure("4L9M", pdb_file)

chain_id = "A"

residue_names = []

for model in structure:
    chain = model[chain_id]

    for res in chain:
        resnum = res.get_id()[1]

        if resnum in pocket_residues:
            residue_names.append({
                "chain": chain_id,
                "residue_number": resnum,
                "residue_name": res.get_resname()
            })

residue_names

[{'chain': 'A', 'residue_number': 219, 'residue_name': 'LYS'},
 {'chain': 'A', 'residue_number': 220, 'residue_name': 'SER'},
 {'chain': 'A', 'residue_number': 223, 'residue_name': 'ARG'},
 {'chain': 'A', 'residue_number': 246, 'residue_name': 'ARG'},
 {'chain': 'A', 'residue_number': 249, 'residue_name': 'ALA'},
 {'chain': 'A', 'residue_number': 250, 'residue_name': 'LEU'},
 {'chain': 'A', 'residue_number': 414, 'residue_name': 'THR'},
 {'chain': 'A', 'residue_number': 417, 'residue_name': 'LEU'},
 {'chain': 'A', 'residue_number': 418, 'residue_name': 'ASP'},
 {'chain': 'A', 'residue_number': 420, 'residue_name': 'TYR'},
 {'chain': 'A', 'residue_number': 461, 'residue_name': 'LYS'},
 {'chain': 'A', 'residue_number': 462, 'residue_name': 'PRO'},
 {'chain': 'A', 'residue_number': 463, 'residue_name': 'ASP'},
 {'chain': 'A', 'residue_number': 466, 'residue_name': 'THR'},
 {'chain': 'A', 'residue_number': 467, 'residue_name': 'ILE'},
 {'chain': 'A', 'residue_number': 470, 'residue_name': 

In [ ]:
df_pocket = pd.DataFrame(residue_names)

df_pocket

,chain,residue_number,residue_name
0,A,219,LYS
1,A,220,SER
2,A,223,ARG
3,A,246,ARG
4,A,249,ALA
5,A,250,LEU
6,A,414,THR
7,A,417,LEU
8,A,418,ASP
9,A,420,TYR


In [ ]:
hydrophobic = {"ALA", "VAL", "LEU", "ILE", "MET", "PRO"}
aromatic = {"PHE", "TYR", "TRP"}
polar = {"SER", "THR", "ASN", "GLN", "CYS"}
positive = {"LYS", "ARG", "HIS"}
negative = {"ASP", "GLU"}
special = {"GLY"}

def classify_residue(res):
    if res in hydrophobic:
        return "hidrofóbico"
    elif res in aromatic:
        return "aromático"
    elif res in polar:
        return "polar"
    elif res in positive:
        return "básico/carga positiva"
    elif res in negative:
        return "ácido/carga negativa"
    elif res in special:
        return "especial"
    else:
        return "otro"

df_pocket["tipo"] = df_pocket["residue_name"].apply(classify_residue)

df_pocket

,chain,residue_number,residue_name,tipo
0,A,219,LYS,básico/carga positiva
1,A,220,SER,polar
2,A,223,ARG,básico/carga positiva
3,A,246,ARG,básico/carga positiva
4,A,249,ALA,hidrofóbico
5,A,250,LEU,hidrofóbico
6,A,414,THR,polar
7,A,417,LEU,hidrofóbico
8,A,418,ASP,ácido/carga negativa
9,A,420,TYR,aromático


In [ ]:
n_total = len(df_pocket)

n_hidrofobicos = df_pocket["tipo"].isin(["hidrofóbico", "aromático"]).sum()

n_polares = df_pocket["tipo"].isin(["polar"]).sum()

porcentaje_hidrofobico = 100 * n_hidrofobicos / n_total
porcentaje_polar = 100 * n_polares / n_total

print(f"Residuos hidrofóbicos/aromáticos: {n_hidrofobicos}/{n_total}")
print(f"Porcentaje hidrofóbico/aromático: {porcentaje_hidrofobico:.1f}%")
print("porcentaje polar:", porcentaje_polar)

Residuos hidrofóbicos/aromáticos: 6/17
Porcentaje hidrofóbico/aromático: 35.3%
porcentaje polar: 17.647058823529413


In [ ]:


# Cambiá estos valores por los de tu tabla Foldseek/TM-align
target_start = 51
target_end = 177

# Ver qué residuos del pocket caen dentro del rango alineado
residuos_dentro = [
    r for r in pocket_residues
    if target_start <= r <= target_end
]

residuos_fuera = [
    r for r in pocket_residues
    if not (target_start <= r <= target_end)
]

porcentaje_dentro = 100 * len(residuos_dentro) / len(pocket_residues)

print("Residuos dentro de la región alineada:", residuos_dentro)
print("Residuos fuera de la región alineada:", residuos_fuera)
print(f"Porcentaje dentro: {porcentaje_dentro:.1f}%")

Residuos dentro de la región alineada: []
Residuos fuera de la región alineada: [219, 220, 223, 246, 249, 250, 414, 417, 418, 420, 461, 462, 463, 466, 467, 470, 473]
Porcentaje dentro: 0.0%
